# all-reduce-eval-metrics — worked example 2: All-reduce a per-class correct-count vector for balanced accuracy

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `all-reduce-eval-metrics`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Balanced accuracy needs per-class correct and per-class total counts, not just a single scalar. Each rank holds a length-`C` vector of correct counts and a length-`C` vector of class totals for its shard. A single `all_reduce(SUM)` over a stacked `2 x C` tensor synchronizes both vectors at once, after which every rank divides element-wise to recover the per-class accuracy.

## Worked solution

**Step 1 — set up the problem.** With `C=3` classes and `world_size=2`, rank 0 saw correct counts `[5, 3, 4]` out of totals `[6, 4, 5]`, and rank 1 saw correct `[2, 7, 1]` out of totals `[3, 9, 2]`. These are vectors, not scalars, so we can't reuse the length-2 packing from the scalar case directly — we stack two rows.

**Step 2 — stack correct and total into a 2xC tensor.** Row 0 = correct vector, row 1 = total vector. Stacking keeps everything in one tensor so a single `all_reduce` handles all `2*C` numbers. `t.stack([correct, total])` produces shape `(2, C)`.

**Step 3 — one SUM all_reduce over the matrix.** `all_reduce` is element-wise and shape-agnostic, so reducing a `(2, C)` tensor sums each of the `2*C` entries across ranks. Afterward row 0 holds global per-class correct `[7, 10, 5]` and row 1 holds global per-class totals `[9, 13, 7]`.

**Step 4 — element-wise divide and mean.** Per-class accuracy is `row0 / row1 = [0.778, 0.769, 0.714]`. Balanced accuracy is the unweighted mean of those per-class accuracies, `~0.754`. Using float tensors avoids integer-division surprises and matches the `all_reduce` dtype requirement.

**Step 5 — verify against direct totals.** The mock sums the rank tensors in-process, so the stacked global tensor exactly equals summing the raw vectors — confirming the reduce did the bookkeeping correctly.

In [ ]:
class MockDist:
    class ReduceOp:
        SUM = 'sum'
    def __init__(self, rank_tensors):
        self.rank_tensors = rank_tensors
    def all_reduce(self, tensor, op=ReduceOp.SUM):
        total = t.zeros_like(self.rank_tensors[0])
        for rt in self.rank_tensors:
            total += rt
        tensor.copy_(total)

def balanced_accuracy(dist_module, local_correct, local_total):
    packed = t.stack([local_correct.float(), local_total.float()])  # (2, C)
    dist_module.all_reduce(packed, op=dist_module.ReduceOp.SUM)
    per_class = packed[0] / packed[1]
    return per_class, per_class.mean().item()

t.manual_seed(0)
r0_correct, r0_total = t.tensor([5, 3, 4]), t.tensor([6, 4, 5])
r1_correct, r1_total = t.tensor([2, 7, 1]), t.tensor([3, 9, 2])
rank_tensors = [
    t.stack([r0_correct.float(), r0_total.float()]),
    t.stack([r1_correct.float(), r1_total.float()]),
]
mock = MockDist(rank_tensors)
per_class, bal = balanced_accuracy(mock, r0_correct, r0_total)
print('per-class acc:', [round(x, 4) for x in per_class.tolist()])
print('balanced accuracy:', round(bal, 6))